# 09 — Auditoria: os números são confiáveis?

Corresponde à seção **§13** de `docs/DOCUMENTACAO.md`

O projeto passou por duas auditorias. Este notebook **executa** as verificações finais, as mesmas que `python run_all.py` roda no fim:

1. **Recálculo independente:** os filtros, índices e correlações das quatro perguntas são reimplementados **sem usar** as funções de análise (`utils.py`) e comparados com os 93 resultados oficiais
2. **Documento × resultados:** os números-chave dos resultados oficiais precisam aparecer no texto da documentação e do README
3. **Data dos arquivos:** nenhuma tabela ou gráfico pode ser mais antigo que os dados de que depende

No final, o registro do que foi corrigido em cada auditoria.

In [1]:
import sys, json, warnings
from pathlib import Path

# encontra a raiz do projeto (pasta que contém src/config.py), venha o notebook de onde vier
ROOT = Path.cwd().resolve()
while not (ROOT / "src" / "config.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, Image

import config as C
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 80)
print("Projeto:", ROOT)
import auditoria as AUD

Projeto: C:\Users\braia\OneDrive\Documentos\GitHub\Datasets-Beready


## 1. Recálculo independente

In [2]:
AUD.consistencia()
ri = AUD.OUT["consistencia"]["recalculo_independente"]
print(f"{ri['conferem']}/{ri['total']} verificações conferem")
print("Saídas mais antigas que os dados processados:", AUD.OUT["consistencia"]["saidas_mais_antigas_que_os_dados_processados"] or "nenhuma")
det = pd.DataFrame(ri["detalhe"])
det.head(40)

93/93 verificações conferem
Saídas mais antigas que os dados processados: nenhuma


,verificacao,oficial,recalculado,confere
0,ARES Q1 n,141.000000,141.0000,True
1,ARES Q1 ρ interações×nota pós,0.078900,0.0789,True
2,ARES Q1 ρ parcial (pré),0.057600,0.0576,True
3,ARES Q5 ρ interações×evolução,0.001000,0.0010,True
4,ARES Q4 ρ corrigido — Acesso (login),0.146100,0.1461,True
5,ARES Q4 ρ corrigido — Avaliação do texto (estrelas),-0.072700,-0.0727,True
6,ARES Q4 ρ corrigido — Leitura de textos,0.314300,0.3143,True
7,ARES Q4 ρ corrigido — Navegação entre páginas,0.242100,0.2421,True
8,ARES Q4 ρ corrigido — Outros (diálogos/botões),-0.004500,-0.0045,True
9,ARES Q4 ρ corrigido — Responder questões/tarefas,-0.024000,-0.0240,True


In [3]:
det.tail(53)

,verificacao,oficial,recalculado,confere
40,ARES Q10 ρ parcial (pré) — Visualização de feedback/nota,-0.042700,-0.0427,True
41,ARES Q10 % eventos — Visualização de feedback/nota,0.550000,0.5515,True
42,ARES Q10 ρ parcial (pré) — Vocabulário (palavras e traduções),0.171600,0.1716,True
43,ARES Q10 % eventos — Vocabulário (palavras e traduções),11.810000,11.8056,True
44,ARES Q10 p Holm — Acesso (login),1.000000,1.0000,True
45,ARES Q10 p Holm — Avaliação do texto (estrelas),1.000000,1.0000,True
46,ARES Q10 p Holm — Leitura de textos,1.000000,1.0000,True
47,ARES Q10 p Holm — Navegação entre páginas,1.000000,1.0000,True
48,ARES Q10 p Holm — Outros (diálogos/botões),1.000000,1.0000,True
49,ARES Q10 p Holm — Responder questões/tarefas,1.000000,1.0000,True


In [4]:
assert not ri["falhas"], ri["falhas"]
print("✅ Todos os resultados oficiais foram reproduzidos por um cálculo independente.")

✅ Todos os resultados oficiais foram reproduzidos por um cálculo independente.


## 2. Documento × resultados

In [5]:
AUD.documentacao()
doc = pd.DataFrame(AUD.OUT["documentacao"]["verificacoes"])
display(doc)
assert not AUD.OUT["documentacao"]["faltando"]
print("✅ Todos os números-chave estão no texto da documentação (e no README, quando aplicável).")

,valor,esperado,no_documento,no_readme
0,ARES Q1 ρ,"+0,08",True,None
1,EdNet Q1 n,76.252,True,None
2,EdNet Q1 ρ,"+0,15",True,True
3,EdNet Q1 ρ parcial,"+0,14",True,None
4,ARES Q4 leitura,"+0,31",True,True
5,ARES Q4 navegação,"+0,24",True,None
6,ARES Q4 feedback,"+0,14",True,None
7,ARES Q4 Holm turmas,"0,54",True,None
8,EdNet Q4 aulas,"+0,24",True,True
9,EdNet Q4 explicações,"+0,06",True,None


✅ Todos os números-chave estão no texto da documentação (e no README, quando aplicável).


## 3. Reprodução do notebook antigo (Q1) e das fórmulas da Q5

In [6]:
AUD.q1()
pd.DataFrame(AUD.OUT["Q1_decomposicao"]).T.iloc[:4][["n", "pearson_r", "pearson_p", "spearman_rho", "spearman_p"]]

,n,pearson_r,pearson_p,spearman_rho,spearman_p
"A_notebook_antigo (todos os eventos, com duplicatas, nota 0–12)",141,0.1178,0.1642,0.1005,0.2357
B = A + nota oficial 0–11 (sem Q9),141,0.0939,0.268,0.0837,0.3237
C = B + sem duplicatas,141,0.121,0.1528,0.0889,0.2945
D = C + só eventos de aprendizagem (DOCUMENTAÇÃO),141,0.1153,0.1733,0.0789,0.3524


In [7]:
AUD.q5()
ed = AUD.OUT["Q5_ednet"]
pd.DataFrame({"fórmula antiga": {k: ed["original"][k] for k in ["n", "inicio", "fim", "evolucao_media", "pct_melhorou"]},
              "fórmula corrigida": {k: ed["corrigida"][k] for k in ["n", "inicio", "fim", "evolucao_media", "pct_melhorou"]}})

,fórmula antiga,fórmula corrigida
n,46635.0000,42720.0000
inicio,0.5855,0.6362
fim,0.6441,0.6462
evolucao_media,0.0586,0.0100
pct_melhorou,58.5000,47.2000


## 4. O que foi corrigido (resumo da §13)

| # | Problema | Efeito na conclusão | Onde ver |
|---|---|---|---|
| 1 | **Q4:** índice de engajamento com o volume dos dois lados da correlação (associação mecânica) | “Ver feedback marca os engajados” não se sustenta; fica “estudar conteúdo” | notebook 05 |
| 2 | **Q5 (EdNet):** janela inicial com 36% de questões do diagnóstico | Evolução média +5,9 → **+1,0 p.p.**; associação continua desprezível | notebook 06 |
| 3 | **Q10:** sem correção de Holm, sem controle de volume, denominadores misturados | Vocabulário deixa de ser significativo; prática usada por 89% dos ativos | notebook 07 |
| 4 | **Q1 (ARES):** Pearson 0,118 × Spearman 0,08 | Mesmos 141 alunos; a diferença é de método; ambos não significativos | notebook 04 |
| 5 | Comparação ignorava significância; linguagem causal | Regra refeita; recomendações viraram hipóteses | notebook 08 |
| 6 | **Q1 (EdNet):** acurácia misturava diagnóstico e prática | ρ 0,36 → **0,15** | notebook 04 |
| 7 | Projeto não rodava em outro computador | Caminhos portáveis; EdNet aceito como zip **ou** pasta | notebook 01 |
| 8 | Arredondamento do índice; graus de liberdade da correlação parcial | Mudanças na 3ª casa decimal | notebooks 04 e 05 |

## Como rodar tudo de novo

```bash
python run_all.py                        # pipeline completo (regenera dados, tabelas e gráficos)
python run_all.py --skip-ednet-extract   # sem refazer a extração do EdNet (~1–2 min)
```

Depois, abra os notebooks na ordem (01 → 09) e use **Run All**.